In [ ]:
# Install dependencies
!pip install transformers datasets sacrebleu kagglehub --quiet

import os, random
import torch
from datasets import Dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from sacrebleu import corpus_bleu, corpus_chrf
import kagglehub
import numpy as np

# ============================
# 1. Load Dataset
# ============================
dataset_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset")

with open(os.path.join(dataset_path, "train.en"), encoding="utf-8") as f:
    english = [l.strip() for l in f][:2000]

with open(os.path.join(dataset_path, "train.ml"), encoding="utf-8") as f:
    malayalam = [l.strip() for l in f][:2000]

pairs = [(en, ml) for en, ml in zip(english, malayalam) if en and ml]
english, malayalam = zip(*pairs)

# Train / Eval / Test split
random.seed(42)
indices = list(range(len(english)))
random.shuffle(indices)

train_en = [english[i] for i in indices[:1200]]
train_ml = [malayalam[i] for i in indices[:1200]]
eval_en  = [english[i] for i in indices[1200:1400]]
eval_ml  = [malayalam[i] for i in indices[1200:1400]]
test_en  = [english[i] for i in indices[1400:1600]]
test_ml  = [malayalam[i] for i in indices[1400:1600]]

print("Loaded data: Train=", len(train_en), "Eval=", len(eval_en), "Test=", len(test_en))

# ============================
# 2. Load MarianMT model
# ============================
model_name = "Helsinki-NLP/opus-mt-en-ml"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model loaded on", device)

# ============================
# 3. Preprocessing
# ============================
def preprocess(batch):
    inputs = tokenizer(batch["en"], truncation=True, padding="max_length", max_length=128)
    outputs = tokenizer(batch["ml"], truncation=True, padding="max_length", max_length=128)

    # Replace PAD tokens with -100 in labels
    labels = []
    for seq in outputs["input_ids"]:
        labels.append([(t if t != tokenizer.pad_token_id else -100) for t in seq])

    inputs["labels"] = labels
    return inputs

train_ds = Dataset.from_dict({"en": train_en, "ml": train_ml}).map(
    preprocess, batched=True, remove_columns=["en", "ml"]
)
eval_ds = Dataset.from_dict({"en": eval_en, "ml": eval_ml}).map(
    preprocess, batched=True, remove_columns=["en", "ml"]
)

# ============================
# 4. Compute BLEU
# ============================
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": bleu.score}

# ============================
# 5. Training Setup
# ============================
print("\nCreating training arguments...")

model.gradient_checkpointing_enable()
model.config.use_cache = False  # Required with checkpointing to avoid memory spikes

args = Seq2SeqTrainingArguments(
    output_dir="nllb_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,   # MUCH smaller
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,   # Effective batch size = 4
    learning_rate=3e-5,
    predict_with_generate=True,
    logging_steps=50,
    save_strategy="no",
    eval_strategy="no",
    fp16=False,
    bf16=True,   # MORE STABLE on Colab GPUs
    report_to=[],
    generation_max_length=64,  # smaller = lower memory
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)


collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Training starting…")
trainer.train()
print("Training finished!")

# ============================
# 6. Final Evaluation on Test Set
# ============================
model.eval()
predictions = []

for i in range(0, len(test_en), 16):
    batch = tokenizer(test_en[i:i+16], return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        generated = model.generate(**batch, max_length=128, num_beams=4)

    preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
    predictions.extend(preds)

bleu = corpus_bleu(predictions, [list(test_ml)])
chrf = corpus_chrf(predictions, [list(test_ml)])

print("\n==== FINAL RESULTS ====")
print("BLEU:", bleu.score)
print("chrF:", chrf.score)

# ============================
# 7. Save outputs
# ============================
os.makedirs("marian_results", exist_ok=True)
with open("marian_results/predictions.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(predictions))

with open("marian_results/references.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_ml))

model.save_pretrained("marian_results/fine_tuned_model")
tokenizer.save_pretrained("marian_results/fine_tuned_model")

print("\nSaved model + predictions!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.4 MB/s eta 0:00:00


100%|██████████| 407M/407M [00:20<00:00, 21.2MB/s]

Extracting files...


Loaded data: Train= 1200 Eval= 200 Test= 200


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

/tmp/ipython-input-756825082.py:120: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training starting…


Epoch,Training Loss,Validation Loss,Bleu
1,2.799900,2.286436,2.636413
2,0.967400,1.023014,15.469700
3,0.784200,0.930143,16.798080
4,0.639600,0.907046,19.298765
5,0.592900,0.923021,18.191397
6,0.516000,0.946409,19.643854
7,0.485200,0.940036,20.461448
8,0.442900,0.965403,20.894120
9,0.431500,0.973578,20.364687
10,0.415900,0.977520,20.266149


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[24660]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Training finished!

==== FINAL RESULTS ====
BLEU: 0.07951187676108801
chrF: 1.3345605411714638

Saved model + predictions!


In [ ]:
!pip install transformers sacrebleu --quiet

import torch
from transformers import MarianMTModel, MarianTokenizer
from sacrebleu import corpus_bleu, corpus_chrf

# ----------------------------------
# 1. Load MarianMT English → Malayalam model
# ----------------------------------
model_name = "Helsinki-NLP/opus-mt-en-ml"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded on:", device)

# ----------------------------------
# 2. Use your existing test data
# ----------------------------------
print("Using in-memory test dataset")
print("Number of test samples:", len(test_en))

# ----------------------------------
# 3. Run translations
# ----------------------------------
predictions = []

for i in range(0, len(test_en), 16):
    batch = tokenizer(
        test_en[i:i+16],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **batch,
            num_beams=4,
            max_length=128
        )

    translated = tokenizer.batch_decode(generated, skip_special_tokens=True)
    predictions.extend(translated)

# ----------------------------------
# 4. Evaluate BLEU + chrF
# ----------------------------------
bleu = corpus_bleu(predictions, [list(test_ml)])
chrf = corpus_chrf(predictions, [list(test_ml)])

print("\n===== FINAL EVALUATION =====")
print("BLEU Score:", bleu.score)
print("chrF Score:", chrf.score)

# ----------------------------------
# 5. Print sample translations
# ----------------------------------
print("\n===== SAMPLE TRANSLATIONS =====\n")

for i in range(min(10, len(test_en))):
    print(f"[{i+1}] English:   {test_en[i]}")
    print(f"     Predicted: {predictions[i]}")
    print(f"     Reference: {test_ml[i]}\n")

# ----------------------------------
# 6. Save predictions
# ----------------------------------
with open("predicted_translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(predictions))

print("Saved predictions to predicted_translations.txt")


Model loaded on: cuda
Using in-memory test dataset
Number of test samples: 200

===== FINAL EVALUATION =====
BLEU Score: 4.403362929849364
chrF Score: 31.232591689915722

===== SAMPLE TRANSLATIONS =====

[1] English:   Imitate Their Faith He Stood Up for Pure Worship
     Predicted: അവരുടെ വിശ്വാസം അനുകരിക്കുക
     Reference: അവരുടെ വിശ്വാസം അനുകരിക്കുക

[2] English:   says Mishra.
     Predicted: മിശ്ര പറയുന്നു.
     Reference: ” മിശ്ര പറയുന്നു.

[3] English:   I want to go!
     Predicted: എനിക്ക് പോകണം!
     Reference: എനിക്ക് പോകണം!

[4] English:   Two Indian jawans were killed.
     Predicted: 2 ഇന്ത്യൻ താടിയെല്ലുകൾ കൊല്ലപ്പെട്ടു.
     Reference: രണ്ട് ഇന്ത്യൻ സൈനികർക്ക് പരുക്കേറ്റു.

[5] English:   """Ayurveda is not just a method of treatment."
     Predicted: "ആര്യൂർവേദ വെറുമൊരു ചികിത്സാരീതിയല്ല."
     Reference: ആയുര്‍വേദം ഒരു ചികിത്സാരീതി മാത്രമല്ല.

[6] English:   Many were arrested and jailed.
     Predicted: പലരേയും അറസ്റ്റുചെയ് ത് ജയിലിലടച്ചു.
     Reference: നിരപരാധികളായ

In [ ]:
%reset -f
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
